# Tugas Praktikum 5: Sequential Pattern Analysis

## Mengimpor Package

In [56]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
import zipfile
import requests
import numpy as np
import pandas as pd
from spmf import Spmf

## Mempersiapkan SPMF

In [57]:
if not os.path.exists("spmf.jar"):
    url = "https://www.philippe-fournier-viger.com/spmf/spmf.jar"
    req = requests.get(url, stream=True)
    
    with req:
        with open("spmf.jar", "wb") as f:
            for chunk in req.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    
# PASTIKAN TELAH MEMASANG JAVA 25

## Mengimpor Dataset

In [58]:
if not os.path.exists("Online Retail.xlsx"):
    dataset_url = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
    req = requests.get(dataset_url, stream=True)

    with req:
        with open("download.zip", "wb") as f:
            for chunk in req.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    
    if zipfile.is_zipfile("download.zip"):
        with zipfile.ZipFile('download.zip') as z:
            z.extractall('.')
            print('extracted:', z.namelist())
        
df = pd.read_excel("Online Retail.xlsx")

## Initial EDA

In [59]:
print(df.shape)
display(df.info())
display(df.head())
display(df.describe())
display(df.isna().sum())

(541909, 8)
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


None

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

## Praprocess Data

### Menghapus Baris yang Tidak Relevan atau Tidak Valid

In [60]:
df["InvoiceNo"] = df["InvoiceNo"].map(str)

quantity_mask = df["Quantity"] > 0
unit_price_mask = df["UnitPrice"] > 0
invoice_mask = df["InvoiceNo"].map(lambda x: not x.startswith("C"))
combined_mask = (quantity_mask & invoice_mask & unit_price_mask)
df = df[combined_mask].dropna()

display(df.info())

<class 'pandas.DataFrame'>
Index: 397884 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    397884 non-null  str           
 1   StockCode    397884 non-null  object        
 2   Description  397884 non-null  object        
 3   Quantity     397884 non-null  int64         
 4   InvoiceDate  397884 non-null  datetime64[us]
 5   UnitPrice    397884 non-null  float64       
 6   CustomerID   397884 non-null  float64       
 7   Country      397884 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(2), str(2)
memory usage: 27.3+ MB


None

### Menghapus Fitur yang Tidak Relevan

In [61]:
to_drop = [
    col for col in df.columns if col not in ["CustomerID", "InvoiceDate", "Description", "StockCode"]
]

df = df.drop(columns=to_drop)
display(df.info())

<class 'pandas.DataFrame'>
Index: 397884 entries, 0 to 541908
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   StockCode    397884 non-null  object        
 1   Description  397884 non-null  object        
 2   InvoiceDate  397884 non-null  datetime64[us]
 3   CustomerID   397884 non-null  float64       
dtypes: datetime64[us](1), float64(1), object(2)
memory usage: 15.2+ MB


None

### Memperbaiki Tipe Data Tiap Fitur

In [62]:
for col, datatype in zip(
    df.columns, [str, str, np.datetime64, float]
):
    df[col] = df[col].map(datatype)
    
display(df.info())

<class 'pandas.DataFrame'>
Index: 397884 entries, 0 to 541908
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   StockCode    397884 non-null  str           
 1   Description  397884 non-null  str           
 2   InvoiceDate  397884 non-null  datetime64[us]
 3   CustomerID   397884 non-null  float64       
dtypes: datetime64[us](1), float64(1), str(2)
memory usage: 15.2 MB


None

### Mempersiapkan Dataset dengan Pengelompokkan Berdasarkan CustomerID dan InvoiceDate

In [63]:
# MAP
stock_desc_map = (
    df.assign(Description=df["Description"].str.strip().str.upper())
      .groupby("StockCode")["Description"]
      .agg(lambda s: s.mode().iat[0])
      .to_dict()
)

# GROUP
df = df.drop(columns=["Description"])
baskets = (
    df.groupby(['CustomerID', 'InvoiceDate'])['StockCode']
    .apply(tuple)
    .reset_index()
)

sequences = (
    baskets.sort_values('InvoiceDate')
    .groupby('CustomerID')['StockCode']
    .apply(list)
    .tolist()
)

print("Banyak sequence:", len(sequences))
    
# Contoh output:
# [
#     # CustomerID: 0001
#     [
#         (A, B, C),        # Date 1
#         (B, D),           # Date 2
#         (C, D, E)         # Date 3
#     ],

#     # CustomerID: 0002
#     [
#         (F),              # Date 1
#         (H, J),           # Date 2
#         (A, E, G)         # Date 3
#     ]
# ]

Banyak sequence: 4338


### Menyimpan Dataset ke File Teks untuk SPMF

In [64]:
# MAP
items = sorted({item for seq in sequences for basket in seq for item in basket})
item_to_id = {item: i + 1 for i, item in enumerate(items)}
id_to_item = {i: item for item, i in item_to_id.items()}

print(f"Total unique items mapped: {len(item_to_id)}")

# WRITE
output_filename = "spmf_sequences_input.txt"
with open(output_filename, "w") as f:
    for seq in sequences:
        line_parts = []
        for basket in seq:
            basket_ids = sorted({item_to_id[item] for item in basket})  # int sort + dedupe
            line_parts.append(" ".join(map(str, basket_ids)) + " -1")
        f.write(" ".join(line_parts) + " -2\n")

print(f"Successfully exported sequences to {output_filename}!")

Total unique items mapped: 3665
Successfully exported sequences to spmf_sequences_input.txt!


## Evaluasi Model Sequential Pattern

### Fungsi Pembantu

In [65]:
def parse_spmf_output(output_filepath):
    patterns = []
    with open(output_filepath, 'r') as f:
        for line in f:
            line = line.strip()
            # Skip empty lines or SPMF comments/statistics headers
            if not line or line.startswith('#') or line.startswith('==') or line.startswith('TOTAL'):
                continue
            
            # SPMF output format separates sequence items and support like:
            # "1 2 -1 #SUP: 44"
            if '#SUP:' in line:
                item_part, sup_part = line.split('#SUP:')
                
                # Extract item tokens (consider the SPMF basket delimiter '-1')
                itemset, curr = [], []
                for x in item_part.strip().split():
                    if x == "-1":
                        itemset.append(tuple(sorted(curr)))
                        curr = []
                    else:
                        curr.append(id_to_item[int(x)])

                sup = int(sup_part.strip()) / len(sequences)
                
                patterns.append({'pattern': itemset, 'sup': sup})
                
    return pd.DataFrame(patterns)

def all_unique(pattern):
    items = [i for itemset in pattern for i in itemset]
    return len(items) == len(set(items))

def from_code_to_desc(sequences: list[tuple[str]]):
    res = []
    for seq in sequences:
        curr = []
        for item in seq:
            curr.append(stock_desc_map[item])
        res.append(tuple(curr))
    return res

def canon(df):
    n = len(sequences)
    return {
        (tuple(tuple(sorted(s)) for s in p), round(sup * n))
        for p, sup in zip(df["pattern"], df["sup"])
    }
    
def to_set(df):
    n = len(sequences)
    return {(tuple(p), round(s * n)) for p, s in zip(df["pattern"], df["sup"])}

### Eksekusi SPMF

In [66]:
min_support = 0.01
model = ["PrefixSpan", "GSP"]
pattern_dict = dict()

for model_name in model:
    output_path = f"spmf_{model_name.lower()}_output.txt"
    model_spmf = Spmf(
        model_name,
        spmf_bin_location_dir="./",
        input_filename="spmf_sequences_input.txt",
        output_filename= output_path,
        arguments=[min_support],
        memory=4000       # Allocate Java heap memory (MB)
    )
    model_spmf.run()
    
    # Load results directly into a pandas DataFrame
    model_patterns_df = parse_spmf_output(output_path)
    
    # Sequence must be at least containing two unique items (StockCode).
    sorted_df = model_patterns_df.loc[
        model_patterns_df["pattern"].map(lambda x: True if len(x) > 1 and all_unique(x) else False)
    ].sort_values("sup", ascending=False).reset_index(drop=True)
    display(sorted_df.head(3))
    
    sorted_df["pattern"] = sorted_df["pattern"].map(
        lambda x: from_code_to_desc(x)
    )

    print(f"Top 3 {model_name} Patterns (Descending):")
    for item in sorted_df.head(3).values:
        print(f"- {item[0]}\n  Support: {item[1]}")
        
    pattern_dict[model_name] = model_patterns_df
    
a, b = to_set(pattern_dict["PrefixSpan"]), to_set(pattern_dict["GSP"])
print("\nUkuran himpunan sequence:", len(a), len(b), "| Irisan:", len(a & b))
print("Himpunan hanya di PrefixSpan:", len(a - b), "| Himpunan hanya di GSP:", len(b - a))
for x in list(a - b)[:5]: print("P:", x)
for x in list(b - a)[:5]: print("G:", x)

print("Setelah itemset diurutkan, apakah hasil PrefixSpan dan GSP sama?",
      canon(pattern_dict["PrefixSpan"]) == canon(pattern_dict["GSP"]))

>/home/zico/Documents/py/datmin/Modul-DM-RKA-Zico/Materi/5 - Sequential/tugas-sequential-pattern/spmf.jar
=============  PREFIXSPAN - STATISTICS =============
Total time ~ 34458
 Frequent sequences count : 194580
 Max memory (mb) : 24.2635498046875
 minsup = 44 sequences.
 Pattern count : 194580
 Minsup (as number of sequences)) : 44




,pattern,sup
0,"[(85099B,), (23203,)]",0.054403
1,"[(20725,), (23209,)]",0.049101
2,"[(20725,), (20727,)]",0.048870


Top 3 PrefixSpan Patterns (Descending):
- [('JUMBO BAG RED RETROSPOT',), ('JUMBO BAG VINTAGE DOILY',)]
  Support: 0.054402950668510835
- [('LUNCH BAG RED RETROSPOT',), ('LUNCH BAG VINTAGE DOILY',)]
  Support: 0.04910096818810512
- [('LUNCH BAG RED RETROSPOT',), ('LUNCH BAG  BLACK SKULL.',)]
  Support: 0.04887044721069617
>/home/zico/Documents/py/datmin/Modul-DM-RKA-Zico/Materi/5 - Sequential/tugas-sequential-pattern/spmf.jar
=============  Algorithm GSP v2.67 - STATISTICS =============
 Total time ~ 10450 ms
 Frequent sequences count : 194580
 Max memory (mb):638.0




,pattern,sup
0,"[(85099B,), (23203,)]",0.054403
1,"[(20725,), (23209,)]",0.049101
2,"[(20725,), (20727,)]",0.048870


Top 3 GSP Patterns (Descending):
- [('JUMBO BAG RED RETROSPOT',), ('JUMBO BAG VINTAGE DOILY',)]
  Support: 0.054402950668510835
- [('LUNCH BAG RED RETROSPOT',), ('LUNCH BAG VINTAGE DOILY',)]
  Support: 0.04910096818810512
- [('LUNCH BAG RED RETROSPOT',), ('LUNCH BAG  BLACK SKULL.',)]
  Support: 0.04887044721069617

Ukuran himpunan sequence: 194580 194580 | Irisan: 194580
Himpunan hanya di PrefixSpan: 0 | Himpunan hanya di GSP: 0
Setelah itemset diurutkan, apakah hasil PrefixSpan dan GSP sama? True
